In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


掛載雲端硬碟後，您可以像這樣載入檔案 (例如 CSV 檔案)：

```python
import pandas as pd

# 請將 'your_file.csv' 替換為您的檔案在 Google 雲端硬碟中的實際路徑
# 例如：'/content/drive/MyDrive/my_folder/my_data.csv'
file_path = '/content/drive/MyDrive/your_file.csv'
df = pd.read_csv(file_path)

display(df.head())
```

In [7]:
import os

folder_path = '/content/drive/MyDrive/Colab Notebooks/紙袋檢測'

if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"資料夾 '{folder_path}' 已創建。")
else:
    print(f"資料夾 '{folder_path}' 已存在。")

資料夾 '/content/drive/MyDrive/Colab Notebooks/紙袋檢測' 已存在。


In [8]:
import os

folder_to_check = '/content/drive/MyDrive/Colab Notebooks/紙袋檢測'

if os.path.exists(folder_to_check) and os.path.isdir(folder_to_check):
    contents = os.listdir(folder_to_check)
    if contents:
        print(f"資料夾 '{folder_to_check}' 中包含以下內容：")
        for item in contents:
            print(f"- {item}")
    else:
        print(f"資料夾 '{folder_to_check}' 為空。")
else:
    print(f"資料夾 '{folder_to_check}' 不存在或不是一個目錄。")

資料夾 '/content/drive/MyDrive/Colab Notebooks/紙袋檢測' 中包含以下內容：
- PaperBag-Defect.v1i.yolov11.zip
- README.dataset.txt
- README.roboflow.txt
- data.yaml
- train
- valid


In [9]:
import os
import zipfile

zip_file_name = 'PaperBag-Defect.v1i.yolov11.zip'
folder_path = '/content/drive/MyDrive/Colab Notebooks/紙袋檢測'
zip_file_path = os.path.join(folder_path, zip_file_name)
extraction_path = folder_path # Extract to the same folder

if os.path.exists(zip_file_path):
    try:
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall(extraction_path)
        print(f"檔案 '{zip_file_name}' 已成功解壓縮到 '{extraction_path}'。")
        # List contents after extraction to confirm
        print(f"解壓縮後 '{extraction_path}' 中的內容：")
        for item in os.listdir(extraction_path):
            print(f"- {item}")
    except zipfile.BadZipFile:
        print(f"錯誤：'{zip_file_name}' 不是一個有效的 ZIP 檔案。")
    except Exception as e:
        print(f"解壓縮檔案 '{zip_file_name}' 時發生錯誤：{e}")
else:
    print(f"檔案 '{zip_file_name}' 在 '{folder_path}' 中不存在。")

檔案 'PaperBag-Defect.v1i.yolov11.zip' 已成功解壓縮到 '/content/drive/MyDrive/Colab Notebooks/紙袋檢測'。
解壓縮後 '/content/drive/MyDrive/Colab Notebooks/紙袋檢測' 中的內容：
- 紙袋檢測.ipynb
- PaperBag-Defect.v1i.yolov11.zip
- README.dataset.txt
- README.roboflow.txt
- data.yaml
- train
- valid


In [10]:
# 安裝 YOLOv8 套件
!pip install ultralytics

# 確認安裝是否成功 (如果成功，會跑出一些版本資訊)
import ultralytics
ultralytics.checks()

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.1/112.6 GB disk)


In [11]:
from ultralytics import YOLO

# 1. 載入模型
model = YOLO('yolov8n.pt')

# 2. 開始訓練
# 注意：我們把您的路徑放進去了，Python 會自動處理空格和中文
results = model.train(
    data='/content/drive/MyDrive/Colab Notebooks/紙袋檢測/data.yaml',
    epochs=150,
    imgsz=640,
    project='runs/detect',
    name='train_paperbag'
)

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Colab Notebooks/紙袋檢測/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train_paperbag, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto

In [12]:
import random
import glob
import os
from ultralytics import YOLO
from IPython.display import Image, display

# --- 1. 自動尋找最新的模型 (best.pt) ---
weights_list = glob.glob('runs/detect/*/weights/best.pt')
if not weights_list:
    print("❌ 找不到訓練好的模型，請確認訓練是否成功完成。")
else:
    # 找最新的那個模型檔案
    latest_weight = max(weights_list, key=os.path.getctime)
    model = YOLO(latest_weight)
    print(f"✅ 使用模型: {latest_weight}")

    # --- 2. 搜尋圖片並隨機選取 10 張 ---
    # 設定圖片搜尋路徑 (請依實際情況調整)
    base_path = '/content/drive/MyDrive/Colab Notebooks/紙袋檢測'

    # 先找 test 資料夾，沒有的話找全部 jpg
    image_pool = glob.glob(f'{base_path}/**/test/images/*.jpg', recursive=True)
    if not image_pool:
        image_pool = glob.glob(f'{base_path}/**/*.jpg', recursive=True)

    if len(image_pool) == 0:
        print(f"❌ 在 {base_path} 找不到任何圖片。")
    else:
        # 隨機抓取 10 張 (如果不滿 10 張就全抓)
        num_to_pick = min(10, len(image_pool))
        selected_images = random.sample(image_pool, num_to_pick)

        print(f"🎲 隨機選中了 {num_to_pick} 張圖片進行檢測...\n")

        # --- 3. 批量預測並展示 ---
        # save=True 會存檔，conf=0.25 是信心度門檻
        results = model.predict(source=selected_images, save=True, conf=0.25)

        # 顯示結果
        print(f"\n👇👇👇 檢測結果 ({num_to_pick} 張) 👇👇👇")

        # 為了正確顯示剛剛生成的圖，我們去 runs/detect/predict... 資料夾找最新的圖
        # 注意：每次 predict 都會產生一個新的 predict 資料夾 (predict, predict2, predict3...)
        predict_dirs = glob.glob('runs/detect/predict*')
        latest_predict_dir = max(predict_dirs, key=os.path.getctime)

        for img_path in selected_images:
            # 取得檔名
            filename = os.path.basename(img_path)
            # 組合出結果圖的路徑
            result_img_path = os.path.join(latest_predict_dir, filename)

            if os.path.exists(result_img_path):
                display(Image(filename=result_img_path, width=400)) # width可調整顯示大小
            else:
                print(f"找不到結果圖: {filename}")

❌ 找不到訓練好的模型，請確認訓練是否成功完成。


In [13]:
# 顯示訓練結果圖表
Image(filename='runs/detect/train_paperbag2/results.png')

FileNotFoundError: [Errno 2] No such file or directory: 'runs/detect/train_paperbag2/results.png'